# ChemWorld-Bench 全流程演示

这个 notebook 演示当前 ChemWorld-Bench 的完整研究工作流：

1. 初始化 `BatchReactorWorld`。
2. 查看 Chemical World Model foundation 与 physical constitution。
3. 手写一个事件序列实验。
4. 运行官方 baseline 并生成 trajectory JSONL。
5. 读取轨迹、评估指标、重放验证。
6. 运行一个小型 suite 并聚合 leaderboard。
7. 展示 LLM replay agent 如何接入统一事件接口。

它不是教学玩具 demo，而是展示科研级 benchmark 的最小闭环。

## 0. 准备

在项目根目录先安装本地包：

```bash
python -m pip install -e ".[dev]"
```

如果你从 `notebooks/` 目录打开本 notebook，下面的代码会自动识别项目根目录，并把输出写到 `runs/notebook_demo/`。

## Kernel Selection

Use the `Python (ChemWorld)` Jupyter kernel. It should point to this project interpreter:

```text
D:\\Projects\\ChemWorld\\.venv\\Scripts\\python.exe
```

If you see `ModuleNotFoundError: No module named 'gymnasium'`, the notebook is using the wrong kernel. From the project root, run:

```bash
python -m pip install -e ".[dev,notebooks]"
python -m ipykernel install --user --name chemworld --display-name "Python (ChemWorld)"
```

Then switch Jupyter to `Python (ChemWorld)`.


In [1]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Working directory:", Path.cwd())

try:
    import gymnasium  # noqa: F401
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "The active Jupyter kernel does not have ChemWorld dependencies installed. "
        "Select the kernel backed by .venv, or run: "
        "python -m pip install -e '.[dev,notebooks]' && "
        "python -m ipykernel install --user --name chemworld --display-name 'Python (ChemWorld)'"
    ) from exc


Python executable: D:\Projects\ChemWorld\.venv\Scripts\python.exe
Working directory: D:\Projects\ChemWorld\notebooks


In [2]:
import json
from pathlib import Path

import gymnasium as gym
import pandas as pd

import chemworld  # registers BatchReactorWorld
from chemworld.agents.llm import ReplayLLMAgent
from chemworld.data.logging import load_jsonl
from chemworld.eval.leaderboard import aggregate_leaderboard
from chemworld.eval.metrics import evaluate_records
from chemworld.eval.runner import make_agent, run_agent
from chemworld.eval.suite import run_suite
from chemworld.eval.verify import verify_records

ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
OUTPUT_DIR = ROOT / "runs" / "notebook_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("chemworld version:", chemworld.__version__)
print("project root:", ROOT)
print("output dir:", OUTPUT_DIR)

chemworld version: 0.1.0
project root: D:\Projects\ChemWorld
output dir: D:\Projects\ChemWorld\runs\notebook_demo


## 1. 初始化环境

`BatchReactorWorld` 是当前唯一公开环境。每一步都是一个实验操作，例如 `add_solvent`、`heat`、`measure`，而不是一次性黑箱函数调用。

In [3]:
env = gym.make(
    "BatchReactorWorld",
    world_split="public-dev",
    budget=8,
    objective="balanced",
    seed=42,
)

obs, task_info = env.reset(seed=42)
task_summary = {
    "env_id": task_info["env_id"],
    "world_id": task_info["world_id"],
    "world_split": task_info["world_split"],
    "objective": task_info["objective"],
    "budget": task_info["budget"],
    "world_family_version": task_info["world_family_version"],
}
task_summary

{'env_id': 'BatchReactorWorld',
 'world_id': 'BatchReactorWorld:public-dev:public:seed-42',
 'world_split': 'public-dev',
 'objective': 'balanced',
 'budget': 8,
 'world_family_version': 'batch-reactor-foundation'}

In [4]:
pd.DataFrame(task_info["operations"])[["id", "name", "required_fields", "preconditions"]]

,id,name,required_fields,preconditions
0,add_reagent,Add reagent,[amount_mol],[not_terminated]
1,add_solvent,Add solvent,"[volume_L, solvent]",[not_terminated]
2,add_catalyst,Add catalyst,"[catalyst_amount_mol, catalyst]",[not_terminated]
3,heat,Heat,"[target_temperature_K, duration_s, stirring_sp...","[has_volume, has_material]"
4,wait,Wait,"[duration_s, stirring_speed_rpm]","[has_volume, has_material]"
5,sample,Sample,[sample_volume_L],[has_volume]
6,quench,Quench,[],[has_volume]
7,terminate,Terminate,[],[has_material]
8,measure,Measure,[instrument],[]


In [5]:
instrument_rows = []
for instrument_id, spec in task_info["instruments"].items():
    instrument_rows.append(
        {
            "id": instrument_id,
            "name": spec["name"],
            "observable_keys": ", ".join(spec["observable_keys"]),
            "cost": spec["cost"],
            "sample_volume_L": spec["sample_volume_L"],
            "requires_terminated": spec["requires_terminated"],
        }
    )

pd.DataFrame(instrument_rows)

,id,name,observable_keys,cost,sample_volume_L,requires_terminated
0,hplc,HPLC,"yield, selectivity, byproduct_signal",0.080,0.00020,False
1,gc,GC,"byproduct_signal, degradation_warning",0.060,0.00015,False
2,uvvis,UV-vis,"yield, conversion",0.025,0.00005,False
3,final_assay,Final assay,"yield, selectivity, conversion, byproduct_sign...",0.160,0.00030,True


## 2. 查看 Physical Constitution

每个环境状态和观测都要通过可执行 constitution 检查。这里展示的是初始状态的 checklist。

In [6]:
constitution = env.unwrapped.constitution_summary()
print("passed:", constitution["passed"])
print("rules:")
for rule in constitution["rules"]:
    print("-", rule)

constitution_checks = pd.DataFrame(constitution["checks"])
env.close()
constitution_checks.head(12)

passed: True
rules:
- material_conservation
- nonnegative_state
- unit_consistency
- yield_upper_bound
- energy_balance
- observation_non_omniscient
- measurement_has_cost
- action_preconditions
- safety_constraints
- public_private_reproducibility


,name,passed,message,value,tolerance
0,nonnegative:amount:A,True,amount:A=0.0,0.00,5.000000e-07
1,nonnegative:amount:P,True,amount:P=0.0,0.00,5.000000e-07
2,nonnegative:amount:B,True,amount:B=0.0,0.00,5.000000e-07
3,nonnegative:amount:D,True,amount:D=0.0,0.00,5.000000e-07
4,nonnegative:amount:E,True,amount:E=0.0,0.00,5.000000e-07
5,nonnegative:amount:Cat_active,True,amount:Cat_active=0.0,0.00,5.000000e-07
6,nonnegative:amount:Cat_dead,True,amount:Cat_dead=0.0,0.00,5.000000e-07
7,nonnegative:volume_L,True,volume_L=0.0,0.00,5.000000e-07
8,nonnegative:temperature_K,True,temperature_K=298.15,298.15,5.000000e-07
9,nonnegative:pressure_Pa,True,pressure_Pa=101325.0,101325.00,5.000000e-07


## 3. 手写一个实验事件序列

下面是一条完整实验流程：加溶剂、加反应物、加催化剂、升温反应、HPLC 中间测量、等待、终止、最终检测。

In [7]:
manual_actions = [
    {"operation": "add_solvent", "volume_L": 0.030, "solvent": 2},
    {"operation": "add_reagent", "amount_mol": 0.010},
    {"operation": "add_catalyst", "catalyst_amount_mol": 0.00025, "catalyst": 1},
    {
        "operation": "heat",
        "target_temperature_K": 388.0,
        "duration_s": 1500.0,
        "stirring_speed_rpm": 720.0,
    },
    {"operation": "measure", "instrument": "hplc"},
    {"operation": "wait", "duration_s": 600.0, "stirring_speed_rpm": 720.0},
    {"operation": "terminate"},
    {"operation": "measure", "instrument": "final_assay"},
]

manual_env = gym.make(
    "BatchReactorWorld",
    world_split="public-dev",
    budget=len(manual_actions),
    seed=7,
)
manual_env.reset(seed=7)

manual_rows = []
for action in manual_actions:
    observation, reward, terminated, truncated, info = manual_env.step(action)
    row = {
        "step": info["step"],
        "operation": info["operation_type"],
        "instrument": info["instrument"],
        "reward": reward,
        "reward_source": info.get("reward_source"),
        "observed_keys": ", ".join(info.get("observed_keys", [])),
        "leaderboard_score": info.get("leaderboard_score"),
        "yield": float(observation["yield"][0]),
        "selectivity": float(observation["selectivity"][0]),
        "conversion": float(observation["conversion"][0]),
        "cost": float(observation["cost"][0]),
        "risk": float(observation["safety_risk"][0]),
        "precondition_failed": info["constraint_flags"]["precondition_failed"],
        "constitution_failed": info["constraint_flags"]["constitution_failed"],
    }
    manual_rows.append(row)
    if terminated or truncated:
        break

manual_env.close()
pd.DataFrame(manual_rows)

,step,operation,instrument,reward,reward_source,observed_keys,leaderboard_score,yield,selectivity,conversion,cost,risk,precondition_failed,constitution_failed
0,1,add_solvent,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.038400,0.077722,False,False
1,2,add_reagent,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.068400,0.094183,False,False
2,3,add_catalyst,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.248400,0.094183,False,False
3,4,heat,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.260900,0.156874,False,False
4,5,measure,hplc,0.540313,instrument:hplc,"yield, selectivity, byproduct_signal, cost, sa...",NaN,0.839100,0.844464,NaN,0.340900,0.156874,False,False
5,6,wait,None,0.553285,carried_observation_with_public_ledger,"yield, selectivity, byproduct_signal, cost, sa...",NaN,0.839100,0.844464,NaN,0.342567,0.103985,False,False
6,7,terminate,None,0.553285,carried_observation_with_public_ledger,"yield, selectivity, byproduct_signal, cost, sa...",NaN,0.839100,0.844464,NaN,0.342567,0.103985,False,False
7,8,measure,final_assay,0.606183,instrument:final_assay,"yield, selectivity, conversion, byproduct_sign...",0.606183,0.810878,0.811675,0.992067,0.502567,0.103985,False,False


## 4. 运行 baseline 并写出 trajectory

`run_agent` 会调用 agent、执行环境、记录每一步 action/observation/reward/info。如果提供 `output_path`，它会写出标准 JSONL 轨迹。

In [8]:
trajectory_path = OUTPUT_DIR / "random_public_dev_seed11.jsonl"

history = run_agent(
    env_id="BatchReactorWorld",
    agent=make_agent("random"),
    world_split="public-dev",
    budget=8,
    objective="balanced",
    seed=11,
    output_path=trajectory_path,
)

print("steps:", len(history))
print("trajectory:", trajectory_path)

steps: 8
trajectory: D:\Projects\ChemWorld\runs\notebook_demo\random_public_dev_seed11.jsonl


In [9]:
records = load_jsonl(trajectory_path)
trajectory_table = pd.DataFrame(
    {
        "step": record["step"],
        "operation": record["operation_type"],
        "instrument": record["instrument"],
        "reward": record["reward"],
        "reward_source": record.get("reward_source"),
        "observed_keys": ", ".join(record.get("observed_keys", [])),
        "leaderboard_score": record.get("leaderboard_score"),
        "yield": record["observation"]["yield"],
        "selectivity": record["observation"]["selectivity"],
        "conversion": record["observation"]["conversion"],
        "cost": record["observation"]["cost"],
        "risk": record["observation"]["safety_risk"],
        "unsafe": record["constraint_flags"].get("unsafe", False),
        "precondition_failed": record["constraint_flags"].get("precondition_failed", False),
    }
    for record in records
)
trajectory_table

,step,operation,instrument,reward,reward_source,observed_keys,leaderboard_score,yield,selectivity,conversion,cost,risk,unsafe,precondition_failed
0,1,add_solvent,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.018329,0.090322,False,False
1,2,add_reagent,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.056810,0.145957,False,False
2,3,add_catalyst,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.125151,0.145957,False,False
3,4,heat,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.144074,0.151335,False,False
4,5,wait,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.145448,0.147475,False,False
5,6,wait,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.150233,0.145558,False,False
6,7,terminate,None,0.000000,public_ledger_only,"cost, safety_risk, score",NaN,NaN,NaN,NaN,0.150233,0.145558,False,False
7,8,measure,final_assay,0.361334,instrument:final_assay,"yield, selectivity, conversion, byproduct_sign...",0.361334,0.454542,0.467949,1.0,0.310233,0.145558,False,False


## 5. 评估与重放验证

`evaluate_records` 给出 benchmark 指标；`verify_records` 重新执行记录中的动作，确认 observation/reward 可复现。

In [10]:
evaluation = evaluate_records(records)
pd.DataFrame([evaluation.to_dict()]).T.rename(columns={0: "value"})

,value
agent_name,random
env_id,BatchReactorWorld
world_split,public-dev
seed,11
steps,8
final_best_score,0.361334
best_valid_score,0.361334
best_valid_yield,0.454542
area_under_best_score,0.045167
sample_efficiency_step,None


In [11]:
verification = verify_records(records)
verification.to_dict()

{'verified': True, 'checked_steps': 8, 'max_abs_error': 0.0, 'mismatches': []}

## 6. Mini Suite And Leaderboard

This cell runs a small local suite for demonstration. It includes both `public-test` and `private-eval`, so the leaderboard reports `public_private_gap`. Formal experiments should use more seeds, splits, and budget.


In [12]:
suite_results = []
for agent_name in ["random", "scripted_chemistry", "lhs"]:
    suite_results.extend(
        run_suite(
            agent_name=agent_name,
            env_id="BatchReactorWorld",
            world_splits=["public-test", "private-eval"],
            seeds=[0, 1],
            budget=12,
            objective="balanced",
            output_dir=OUTPUT_DIR / "suite" / agent_name,
        )
    )

leaderboard = aggregate_leaderboard(suite_results)
pd.DataFrame(leaderboard)

,agent_name,world_split,runs,mean_total_score,std_total_score,sem_total_score,ci95_total_score_low,ci95_total_score_high,mean_performance,mean_safety_aware_score,public_private_gap,rank
0,latin_hypercube,private-eval,2,0.242994,0.242852,0.171722,0.000000,0.579569,0.277366,0.249969,-0.007128,1
1,latin_hypercube,public-test,2,0.235866,0.103551,0.073222,0.092352,0.379381,0.276591,0.217025,-0.007128,2
2,scripted_chemistry,public-test,2,0.221430,0.044616,0.031549,0.159595,0.283265,0.284805,0.264603,0.038119,3
3,scripted_chemistry,private-eval,2,0.183311,0.146588,0.103654,0.000000,0.386472,0.236631,0.216521,0.038119,4
4,random,private-eval,2,0.102115,0.006069,0.004291,0.093704,0.110526,0.193323,0.055200,-0.045069,5
5,random,public-test,2,0.057046,0.006644,0.004698,0.047838,0.066253,0.103660,0.034277,-0.045069,6


## 7. LLM replay agent 接入方式

核心包不依赖任何在线 LLM API。LLM 结果可以通过 JSONL replay 文件进入同一评测协议。Replay 文件里保存 LLM 规划出的 recipe 参数，agent 会自动展开为事件序列。

In [13]:
replay_path = OUTPUT_DIR / "llm_replay.jsonl"
replay_payload = {
    "temperature": 115.0,
    "time": 1.2,
    "initial_concentration": 0.7,
    "stirring_speed": 680.0,
    "catalyst": 1,
    "solvent": 2,
    "hypothesis": "moderate temperature should limit degradation",
    "rationale": "start with a balanced condition",
}
replay_path.write_text(json.dumps(replay_payload) + "\n", encoding="utf-8")

llm_history = run_agent(
    env_id="BatchReactorWorld",
    agent=ReplayLLMAgent(replay_path),
    world_split="public-dev",
    budget=6,
    objective="balanced",
    seed=21,
)

pd.DataFrame(
    {
        "step": record.step,
        "operation": record.action["operation"],
        "reward": record.reward,
        "score": record.observation["score"],
    }
    for record in llm_history
)

,step,operation,reward,score
0,1,add_solvent,0.00000,0.00000
1,2,add_reagent,0.00000,0.00000
2,3,add_catalyst,0.00000,0.00000
3,4,heat,0.00000,0.00000
4,5,terminate,0.00000,0.00000
5,6,measure,0.02172,0.02172


## 8. 当前闭环已经具备什么

- 统一环境：`BatchReactorWorld`。
- 统一动作接口：事件序列，而不是旧式黑箱 recipe。
- Foundation：ontology、state ledger、constitution、transition kernel、observation kernel。
- Benchmark：JSONL trajectory、metrics、verify、suite、leaderboard。
- Agent：random、scripted、LHS、BO、safe BO、LLM replay adapter。

下一步可以基于这个 notebook 扩展教学 notebook、论文 artifact notebook 或 baseline calibration notebook。